# Step 7 — Content Generation Prompts

**Project:** Prompt Engineering for Clothing Review Analysis  
**Dataset:** Women's Clothing E-Commerce Reviews (`data/reviews.csv`)  
**Goal:** Design, test, and evaluate naive vs. constrained content generation prompts across two business use cases:
1. **Customer Service Recovery:** Crafting tailored, empathetic replies to dissatisfied customer reviews (Rating $\le$ 2).
2. **Marketing Copywriting:** Synthesizing positive customer feedback (Rating $\ge$ 4) into grounded product blurbs without hallucinating features.

## 0. Setup and Data Loading

Resilient file path check so the notebook executes properly from either the project root or the `prompts/` subfolder.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_colwidth", 400)
pd.set_option("display.width", 120)

# Resilient path check
DATA_PATH = (
    Path("data/reviews.csv")
    if Path("data/reviews.csv").exists()
    else Path("..") / "data" / "reviews.csv"
)

df_raw = pd.read_csv(DATA_PATH, index_col=0)
print(f"Loaded: {DATA_PATH.resolve()}")
print(f"Raw dataset shape: {df_raw.shape}")
df_raw.head(3)

Loaded: C:\GamageRecruiters-DataScienceIntern\Month_02\prompt-engineering-task\data\reviews.csv
Raw dataset shape: (23486, 10)


,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,767,33,NaN,Absolutely wonderful - silky and sexy and comfortable,4,1,0,Initmates,Intimate,Intimates
1,1080,34,NaN,"Love this dress! it's sooo pretty. i happened to find it in a store, and i'm glad i did bc i never would have ordered it online bc it's petite. i bought a petite and am 5'8"". i love the length on me- hits just a little below the knee. would definitely be a true midi on someone who is truly petite.",5,1,4,General,Dresses,Dresses
2,1077,60,Some major design flaws,"I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my usual size) but i found this to be outrageously small. so small in fact that i could not zip it up! i reordered it in petite medium, which was just ok. overall, the top half was comfortable and fit nicely, but the bottom half had a very tight under layer and several somewhat cheap...",3,0,0,General,Dresses,Dresses


## 1. Clean Review Text

Apply data cleaning logic identified in previous steps:
1. Drop rows missing `Review Text`.
2. Replace literal `\r\n` Windows line breaks with spaces and trim whitespace.

In [2]:
df = df_raw.copy()

# 1. Drop missing review text records
df = df.dropna(subset=["Review Text"])

# 2. Normalize whitespace and remove literal linebreaks
def normalize_review_text(text):
    text = str(text)
    text = text.replace("\r\n", " ")
    text = text.strip()
    return text

df["Review Text"] = df["Review Text"].map(normalize_review_text)
df = df[df["Review Text"].str.len() > 0].copy()

print(f"Usable clean reviews: {len(df):,}")

Usable clean reviews: 22,641


## 2. Sampling Strategy

We sample distinct datasets for each content generation task:
- **Customer Recovery Targets:** Low-rating reviews (`Rating <= 2`) with length > 100 characters, ensuring sufficient detail for a meaningful response.
- **Marketing Source Sample:** High-rating reviews (`Rating >= 4`) within a single category (`Dresses`) to provide grounded positive themes.

In [3]:
def pick_reply_targets(df, n=2, seed=3):
    """Select n low-rated reviews (>100 chars) that represent clear service recovery opportunities."""
    candidates = df[(df["Rating"] <= 2) & (df["Review Text"].str.len() > 100)]
    return candidates.sample(n=n, random_state=seed)

def pick_marketing_theme_sample(df, class_name="Dresses", n=10, seed=3):
    """Select n high-rated reviews from one product class as source copy material."""
    candidates = df[(df["Class Name"] == class_name) & (df["Rating"] >= 4)]
    return candidates.sample(n=n, random_state=seed)

# Extract samples
reply_targets_df = pick_reply_targets(df, n=2, seed=3)
marketing_sample_df = pick_marketing_theme_sample(df, class_name="Dresses", n=10, seed=3)

print("=== RECOVERY REPLY TARGETS (Rating <= 2) ===")
display(reply_targets_df[["Clothing ID", "Class Name", "Rating", "Recommended IND", "Review Text"]])

print("\n=== MARKETING SOURCE SAMPLE (Dresses, Rating >= 4) ===")
display(marketing_sample_df[["Clothing ID", "Rating", "Review Text"]].head(3))

=== RECOVERY REPLY TARGETS (Rating <= 2) ===


,Clothing ID,Class Name,Rating,Recommended IND,Review Text
15509,862,Knits,2,0,"It's a cute idea, but it's so low cut and the material is very thin. i was hoping to wear it as a cool, flowing summer top, but i'll have to layer it since my whole bra basically hangs out of it. i'm only keeping it because it was such a good price. also the buttons are super tiny and a little hard to fasten."
6926,871,Knits,2,0,"I'm a rather small person--5'2"", about 100 lbs, 32a bust. i typically take a 24p/00p/xxs. knowing how retailer's sizing is a bit all over the place, i ordered this tee in both xxs and xs, planning to return the size that was less preferred. imagine my surprise when i try on one of the tees and i barely can get it on; imagine my further surprise when i realize that the xs is the tee that was so..."



=== MARKETING SOURCE SAMPLE (Dresses, Rating >= 4) ===


,Clothing ID,Rating,Review Text
212,1075,4,"I was so excited about the arrival of my maza dress. much to my surprise the material was not has structured as i thought it would be from the photos. the fit was very tight and did not fall as nicely as i anticipated. because i loved the classic design i decided to give it another chance so i returned it for the next size up and fell in love, the open sleeve was a nice surprise with a touch i..."
13133,1094,5,"I bought this on sale in all 3 colors. it doesn't look like much in the photos, but that's really the point. this is a staple, basic piece. layer on colors and prints, like a printed jacket, scarf or bag. i bought this in wine, along with the sweater knit boots and a bag with a print that ties the colors together. the shape of this dress is great - fitted through the waist and not a lot of ext..."
1404,1074,4,"I kept the size 8 of the dress and had to return the 10 as it was too roomy at the arms and chest (which is great!!!). material is nice and not too much material, hangs nice (its a swing). another win!"


## 3. Formatting Prompt Data

Convert sampled data into clean textual blocks to feed directly into the prompts.

In [4]:
# Select the first recovery target review for the reply experiment
target_review = reply_targets_df.iloc[0]

formatted_reply_data = (
    f"Product: {target_review['Class Name']} (Clothing ID: {target_review['Clothing ID']})\n"
    f"Rating: {target_review['Rating']} out of 5\n"
    f"Customer Review: \"{target_review['Review Text']}\""
)

# Format the 10 positive reviews for marketing synthesis
blurb_lines = ["Here are 10 positive customer reviews for the 'Dresses' collection:\n"]
for i, (_, row) in enumerate(marketing_sample_df.iterrows(), start=1):
    blurb_lines.append(f"Review {i} ({row['Rating']}★): {row['Review Text']}")

formatted_blurb_data = "\n".join(blurb_lines).strip()

print("--- Prompt Data Preview: Recovery Target ---")
print(formatted_reply_data)
print("\n--- Prompt Data Preview: Marketing Source (First 2 Reviews) ---")
print("\n".join(formatted_blurb_data.splitlines()[:4]))

--- Prompt Data Preview: Recovery Target ---
Product: Knits (Clothing ID: 862)
Rating: 2 out of 5
Customer Review: "It's a cute idea, but it's so low cut and the material is very thin. i was hoping to wear it as a cool, flowing summer top, but i'll have to layer it since my whole bra basically hangs out of it. i'm only keeping it because it was such a good price. also the buttons are super tiny and a little hard to fasten."

--- Prompt Data Preview: Marketing Source (First 2 Reviews) ---
Here are 10 positive customer reviews for the 'Dresses' collection:

Review 1 (4★): I was so excited about the arrival of my maza dress. much to my surprise the material was not has structured as i thought it would be from the photos. the fit was very tight and did not fall as nicely as i anticipated. because i loved the classic design i decided to give it another chance so i returned it for the next size up and fell in love, the open sleeve was a nice surprise with a touch if elegance. the front butto

## 4. Prompt Engineering: Customer Service Reply

- `reply_v1_naive`: A generic instruction lacking tone, resolution steps, or length boundaries.
- `reply_v2_improved`: Assigns a retail support persona, mandates acknowledging the exact defect/issue, requires offering an explicit solution (refund/exchange), sets a warm/professional tone, and enforces a strict 80-word limit.

In [5]:
# Customer Reply V1: Naive
reply_v1_naive = "Write a reply to this review."

# Customer Reply V2: Improved
reply_v2_improved = """
You are a senior customer care specialist for a premium women's clothing brand.

Task:
Draft a professional, empathetic reply to the customer review below.

Strict Constraints:
1. Specifically acknowledge and apologize for the exact issue described by the customer (do not use generic corporate apologies).
2. Offer a concrete resolution next step (e.g., immediate replacement, free prepaid exchange, or full refund).
3. Maintain a warm, courteous, and brand-supportive tone.
4. Limit the entire response strictly to under 80 words.
5. Do not include conversational greeting/closing placeholders (e.g., write the actual ready-to-send message).
""".strip()

## 5. Prompt Engineering: Marketing Copy Generation

- `blurb_v1_naive`: A generic request prone to ungrounded claims, exaggerated adjectives, and invented product attributes.
- `blurb_v2_improved`: Sets an e-commerce copywriter persona, restricts claims strictly to positive themes documented in the sample reviews, mandates an exact 3-sentence structure, and includes a retail call-to-action.

In [6]:
# Marketing Blurb V1: Naive
blurb_v1_naive = "Write a product description for this item."

# Marketing Blurb V2: Improved
blurb_v2_improved = """
You are an e-commerce product copywriter.

Task:
Write a compelling marketing product blurb for this dress collection based strictly on the 10 customer reviews provided below.

Strict Constraints:
1. Output exactly 3 sentences.
2. Feature only positive qualities and benefits explicitly highlighted by customers in the reviews (e.g., fabric drape, versatility, flattering fit). Do not invent features or materials not mentioned.
3. Use an upbeat, sophisticated retail tone.
4. The third sentence must be a short, direct call-to-action.
5. Return only the final blurb text without any introductory remarks.
""".strip()

## 6. Copy-Ready Prompt Generation

Run these cells, copy the blocks, and paste them into Claude, ChatGPT, or your preferred LLM interface.

In [7]:
print("=" * 72)
print("COPY FROM HERE — reply_v1 (naive)")
print("=" * 72)
print(f"{reply_v1_naive}\n\n{formatted_reply_data}")
print("=" * 72)
print("COPY TO HERE")
print("=" * 72)

COPY FROM HERE — reply_v1 (naive)
Write a reply to this review.

Product: Knits (Clothing ID: 862)
Rating: 2 out of 5
Customer Review: "It's a cute idea, but it's so low cut and the material is very thin. i was hoping to wear it as a cool, flowing summer top, but i'll have to layer it since my whole bra basically hangs out of it. i'm only keeping it because it was such a good price. also the buttons are super tiny and a little hard to fasten."
COPY TO HERE


In [8]:
print("=" * 72)
print("COPY FROM HERE — reply_v2 (improved)")
print("=" * 72)
print(f"{reply_v2_improved}\n\n{formatted_reply_data}")
print("=" * 72)
print("COPY TO HERE")
print("=" * 72)

COPY FROM HERE — reply_v2 (improved)
You are a senior customer care specialist for a premium women's clothing brand.

Task:
Draft a professional, empathetic reply to the customer review below.

Strict Constraints:
1. Specifically acknowledge and apologize for the exact issue described by the customer (do not use generic corporate apologies).
2. Offer a concrete resolution next step (e.g., immediate replacement, free prepaid exchange, or full refund).
3. Maintain a warm, courteous, and brand-supportive tone.
4. Limit the entire response strictly to under 80 words.
5. Do not include conversational greeting/closing placeholders (e.g., write the actual ready-to-send message).

Product: Knits (Clothing ID: 862)
Rating: 2 out of 5
Customer Review: "It's a cute idea, but it's so low cut and the material is very thin. i was hoping to wear it as a cool, flowing summer top, but i'll have to layer it since my whole bra basically hangs out of it. i'm only keeping it because it was such a good pric

In [9]:
print("=" * 72)
print("COPY FROM HERE — blurb_v1 (naive)")
print("=" * 72)
print(f"{blurb_v1_naive}\n\n{formatted_blurb_data}")
print("=" * 72)
print("COPY TO HERE")
print("=" * 72)

COPY FROM HERE — blurb_v1 (naive)
Write a product description for this item.

Here are 10 positive customer reviews for the 'Dresses' collection:

Review 1 (4★): I was so excited about the arrival of my maza dress. much to my surprise the material was not has structured as i thought it would be from the photos. the fit was very tight and did not fall as nicely as i anticipated. because i loved the classic design i decided to give it another chance so i returned it for the next size up and fell in love, the open sleeve was a nice surprise with a touch if elegance. the front buttons added a bit of old charm and feminine
appeal. while the material was not v
Review 2 (5★): I bought this on sale in all 3 colors. it doesn't look like much in the photos, but that's really the point. this is a staple, basic piece. layer on colors and prints, like a printed jacket, scarf or bag. i bought this in wine, along with the sweater knit boots and a bag with a print that ties the colors together. the sh

In [10]:
print("=" * 72)
print("COPY FROM HERE — blurb_v2 (improved)")
print("=" * 72)
print(f"{blurb_v2_improved}\n\n{formatted_blurb_data}")
print("=" * 72)
print("COPY TO HERE")
print("=" * 72)

COPY FROM HERE — blurb_v2 (improved)
You are an e-commerce product copywriter.

Task:
Write a compelling marketing product blurb for this dress collection based strictly on the 10 customer reviews provided below.

Strict Constraints:
1. Output exactly 3 sentences.
2. Feature only positive qualities and benefits explicitly highlighted by customers in the reviews (e.g., fabric drape, versatility, flattering fit). Do not invent features or materials not mentioned.
3. Use an upbeat, sophisticated retail tone.
4. The third sentence must be a short, direct call-to-action.
5. Return only the final blurb text without any introductory remarks.

Here are 10 positive customer reviews for the 'Dresses' collection:

Review 1 (4★): I was so excited about the arrival of my maza dress. much to my surprise the material was not has structured as i thought it would be from the photos. the fit was very tight and did not fall as nicely as i anticipated. because i loved the classic design i decided to give 

## Output — reply_v1 

Hi there, thank you for sharing your feedback with us! We're glad to hear you loved the cute style and great price, but we're sorry to learn that the fit and material didn't quite meet your expectations for a breezy summer top. We appreciate your helpful notes regarding the low-cut neckline, thin fabric, and tiny buttons, and we will certainly share this feedback with our design and production teams. Thank you for choosing to keep it and finding a way to layer it—if there is ever anything else we can do to help, please don't hesitate to reach out!

## Output — reply_v2 

We are so sorry the knit top was too low-cut, featured unexpectedly thin material, and had difficult tiny buttons. We completely understand how frustrating it is when a piece requires unexpected layering. To make this right, we would love to issue a full refund or send a complimentary alternative style—whichever you prefer. Please reach out to our support team directly so we can process this for you immediately!

## Output — blurb_v1 

## Product Description: The Ultimate Versatile Statement Dress

Discover your new go-to wardrobe staple with our stunning collection of dresses, designed to blend classic charm with effortless modern style. Loved by customers for their unique details, flattering silhouettes, and beautiful movement, these dresses are crafted to make you look and feel your absolute best.

### Key Features & Design Details

* **Charming Accents:** Thoughtful design elements like elegant open sleeves, vintage-inspired front buttons, and playful contrast under-layers add a touch of feminine sophistication.
* **Flattering Silhouettes:** Ranging from sleek, waist-defining cuts to lovely swing silhouettes that offer effortless flow and perfect movement for any occasion.
* **Versatile Styling:** Designed to be a foundational basic that easily transitions from casual daywear layered with jackets and scarves to statement evening wear paired with your favorite heels.
* **Quality Construction:** Made from comfortable, high-quality materials that range from cozy knits to structured fabrics, ensuring a beautiful drape and lasting wear.

### Customer Notes for the Best Fit

* **Sizing Guidance:** Many customers note that these styles offer a tailored or fitted shape through the waist and shoulders. If you prefer a more relaxed fit in the arms or chest, sizing up may give you the ideal look.
* **Length Variety:** Depending on your height, lengths vary across styles from chic midi cuts to playful mid-thigh or ankle-length uneven hems, making them easy to dress up or tailor to your personal preference.

Elevate your rotation with a piece that is as unique, intriguing, and vibrant as you are!

## Output — blurb_v2 

Elevate your everyday wardrobe with our stunning dress collection, featuring a flattering fitted waist, charming feminine details, and a graceful swing silhouette. Crafted from high-quality, comfortable fabrics with beautiful drapes and cheerful colors, these versatile pieces effortlessly transition from day to night. Shop the collection today to find your new favorite staple style!

## Comparison notes

### 1. Customer Support Reply (reply_v1 vs. reply_v2)
- **Risks/Deficiencies in v1:**
  - **No resolution offered:** The response passively thanks the customer for keeping the flawed garment and finding a way to layer it, completely missing the customer-retention opportunity.
  - **Conversational boilerplate:** Required manual editing to strip out introductory conversational filler.
  - **Over-length:** Exceeded 100 words without resolving the core friction point.
- **Improvements in v2:**
  - **Concrete remediation:** Directly offered an immediate choice between a full refund or a complimentary alternative style.
  - **Concise & compliant:** Contained exactly 64 words, well under the 80-word ceiling.
  - **Precision grounding:** Apologized specifically for the low-cut neckline, thin fabric, and tiny buttons rather than using vague corporate platitudes.
- **Prompt Engineering Impact:**
  - Enforcing a dedicated persona, an explicit resolution requirement (refund/exchange), and strict word boundaries prevents brand liability and produces copy ready for production service pipelines.

### 2. Marketing Copy Generation (blurb_v1 vs. blurb_v2)
- **Risks/Deficiencies in v1:**
  - **Format bloat:** Generated a sprawling 220-word product detail page with headings, subheadings, and sizing tips rather than a concise marketing blurb.
  - **Lack of editorial focus:** Too lengthy for social media, ad copy, or quick newsletter spotlights.
- **Improvements in v2:**
  - **Strict structural compliance:** Adhered strictly to the 3-sentence constraint.
  - **Customer-grounded copy:** Synthesized authentic customer sentiment (waist fit, swing drape, comfortable fabrics, cheerful colors) without hallucinating phantom specs.
  - **Action-oriented:** Concluded with a punchy, direct call-to-action ("Shop the collection today to find your new favorite staple style!").
- **Prompt Engineering Impact:**
  - Enforcing sentence limits, tone instructions, and grounding constraints forces the model to distill customer enthusiasm into crisp, high-converting copy without hallucinating unverified product attributes.